<a href="https://colab.research.google.com/github/Claudiopj88/Atividades_ANN/blob/main/Atividade_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz
!rm -r aclImdb/train/unsup

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  42.7M      0  0:00:01  0:00:01 --:--:-- 42.6M


In [2]:
import os, pathlib, shutil, random
base_dir = pathlib.Path("aclImdb")
train_dir = base_dir / "train"
val_dir = base_dir / "val"
test_dir = base_dir / "test"
for category in ["pos", "neg"]:
  os.makedirs(val_dir / category)
  files = os.listdir(train_dir / category)
  random.shuffle(files)
  num_val_samples = int(0.2 * len(files))
  val_files = files[-num_val_samples:]
  for fname in val_files:
    shutil.move(train_dir / category / fname, val_dir / category / fname)

In [3]:
from tensorflow import keras

batch_size = 32
train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=batch_size)
val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/val", batch_size=batch_size)
test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test", batch_size=batch_size)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [4]:
from tensorflow.keras.layers import TextVectorization
max_length = 600
max_tokens=20000
text_vectorization = TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_length)
text_only_train_ds = train_ds.map(lambda x, y: x)
text_vectorization.adapt(text_only_train_ds)
int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)

In [5]:
!pip install transformers

In [6]:
from transformers import pipeline

pipe = pipeline("text-classification", model="lvwerra/distilbert-imdb")

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [7]:
results = pipe(["This is a great movie", "This is a bad movie"])
for result in results:
  print(f"label: {result['label']}, with score: {round(result['score'], 4)}")

label: POSITIVE, with score: 0.9957
label: NEGATIVE, with score: 0.995


In [8]:
from sklearn.metrics import accuracy_score
import numpy as np
import time

y_pred = []
test_labels = []
start = time.time()
# Iterate over the original test_ds which yields (text_string_tensor, label_tensor)
for batch, (texts, targets) in enumerate(test_ds):
  # Process each individual text and its corresponding target label in the batch
  for i in range(len(texts)):
    # Extract the text string and decode it from bytes to utf-8 string
    current_text = texts[i].numpy().decode('utf-8')
    # Pass the string to the pipeline, truncating if necessary
    yp = pipe(current_text, truncation=True, max_length=512) # pipe expects a string or list of strings

    # Get the true label: targets[i] is a scalar tensor. Convert to int, 1 for positive, 0 for negative
    # Assuming `1` corresponds to positive class in `test_ds` (e.g., from text_dataset_from_directory)
    true_label = int(targets[i].numpy() == 1)
    test_labels.append(true_label)

    # Get the predicted label from the pipeline output
    # 'lvwerra/distilbert-imdb' outputs 'POSITIVE' or 'NEGATIVE'
    predicted_label_is_positive = (yp[0]['label'] == 'POSITIVE')
    y_pred.append(predicted_label_is_positive)

  print(f"{batch} {time.time()-start:.2f} s - Accuracy: {accuracy_score(test_labels, y_pred):.4f}")
print("Final Accuracy", accuracy_score(test_labels, y_pred))

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


0 0.76 s - Accuracy: 0.8125
1 1.25 s - Accuracy: 0.8594
2 1.76 s - Accuracy: 0.8958
3 2.19 s - Accuracy: 0.8906
4 2.92 s - Accuracy: 0.8938
5 3.61 s - Accuracy: 0.8906
6 4.13 s - Accuracy: 0.9018
7 4.64 s - Accuracy: 0.9062
8 5.12 s - Accuracy: 0.8958
9 5.80 s - Accuracy: 0.8969
10 6.51 s - Accuracy: 0.9034
11 7.46 s - Accuracy: 0.9062
12 8.31 s - Accuracy: 0.9087
13 9.12 s - Accuracy: 0.9062
14 9.85 s - Accuracy: 0.9042
15 10.35 s - Accuracy: 0.9082
16 10.78 s - Accuracy: 0.9118
17 11.32 s - Accuracy: 0.9149
18 11.80 s - Accuracy: 0.9095
19 12.27 s - Accuracy: 0.9094
20 12.70 s - Accuracy: 0.9122
21 13.20 s - Accuracy: 0.9134
22 13.62 s - Accuracy: 0.9117
23 14.03 s - Accuracy: 0.9115
24 14.53 s - Accuracy: 0.9113
25 14.96 s - Accuracy: 0.9147
26 15.46 s - Accuracy: 0.9144
27 15.94 s - Accuracy: 0.9152
28 16.40 s - Accuracy: 0.9159
29 16.90 s - Accuracy: 0.9177
30 17.35 s - Accuracy: 0.9173
31 17.81 s - Accuracy: 0.9160
32 18.26 s - Accuracy: 0.9167
33 18.74 s - Accuracy: 0.9154
34 19